In [1]:
# MNIST dataset으로 CNN 실습
import tensorflow as tf
print(tf.__version__) # 2.20.0
import numpy as np
import matplotlib.pyplot as plt

# 1) 데이터 준비
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(x_train[0])
print(x_train.shape)  # (60000, 28, 28)

# 채널(channel) 추가 후 정규화 - 흑백인 경우 1
x_train = x_train.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
x_test = x_test.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
print(x_train.shape)
print(x_test.shape)

In [2]:
# 2) 모델 구성
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(16, kernel_size=(3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),     # pool_size=(2, 2) : 2x2 영역에서 최대값을 추출하여 다운샘플링
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Flatten(),      # FCLayer : Fully Connected Layer로 연결하기 위해 1차원으로 변환
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

In [3]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=128,
    callbacks=[es],
    verbose=1
)

In [4]:
# 모델 평가 : 아래 둘의 평가 점수의 차이가 크면 과적합 의심
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}')

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}')

In [5]:
# 모델 저장 및 불러오기
SAVE_PATH = 'cnn1model.keras'
model.save(SAVE_PATH)
print(f'모델 저장 완료 {SAVE_PATH}')

In [6]:
loaded_model=tf.keras.models.load_model(SAVE_PATH)
test_loss, test_acc = loaded_model.evaluate(x_test, y_test, verbose=0)
print(f'불러온 모델의 Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}')

In [7]:
# 분류 예측
idx = 0
x_one = x_test[idx:idx+1]  
y_true = int(y_test[idx])

print(y_true)   # 7

probs = loaded_model.predict(x_one, verbose=0)[0]
y_pred = np.argmax(probs)
print(f'예측 확률: {probs}')
print(f'예측 클래스: {y_pred}')

In [8]:
# 시각화
# 학습 곡선 (정확도/손실)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [10]:
# 혼동행렬로 출력
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred_all = np.argmax(loaded_model.predict(x_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred_all, labels=list(range(10)))

classes = [str(i) for i in range(10)]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()